In [0]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, create_map, lit, upper, trim, coalesce, to_date, lpad, substring
from itertools import chain
import pyspark.sql.functions as F


# Create Spark session
spark = SparkSession.builder.appName("Parking").getOrCreate()

In [0]:
# Load Bronze layer data
bronze_df = spark.table("parking_etl_cluster.default.parking_data_bronze")

In [0]:
bronze_df.display()

In [0]:
bronze_df.count()

#### Data Types Checking


In [0]:
bronze_df.describe()

In [0]:
bronze_df.printSchema()

#### Dropping Unwanted Columns

In [0]:
# Data Cleaning: Drop unnecessary columns
silver_df = bronze_df.drop(
    "Street_Code1",
    "Street_Code2",
    "Street_Code3",
    "Date_First_Observed",
    "Double_Parking_Violation",
    "No_Standing_or_Stopping_Violation",
    "Hydrant_Violation"
)

In [0]:
silver_df.display()

#### Dropping Duplicate Rows by (Summons Number)

In [0]:
# Check before count
print("Before:", silver_df.count())

In [0]:
# Remove duplicate rows based on 'Summons Number'
df_silver_deduped = silver_df.dropDuplicates(["Summons_Number"])

In [0]:
print("After:", df_silver_deduped.count())

In [0]:
silver_df = df_silver_deduped

##### Mapping Vehicle Name


In [0]:
# distinct_vehicle_make = silver_df.select("Vehicle_Make").distinct().rdd.flatMap(lambda x: x).collect()
# print(distinct_vehicle_make)

distinct_vehicle_make = [row["Vehicle_Make"] for row in silver_df.select("Vehicle_Make").distinct().collect()]
print(distinct_vehicle_make)

In [0]:
vehicle_make_map = {
    # Japanese
    "TOYOT": "TOYOTA", "TOYOTA": "TOYOTA", "TOY": "TOYOTA",
    "HOND": "HONDA", "HONDA": "HONDA", "HONFA": "HONDA",
    "NISS": "NISSAN", "NISSA": "NISSAN", "NISSAN": "NISSAN",
    "MAZDA": "MAZDA", "MAZ": "MAZDA", "SUBAR": "SUBARU", "SUBA": "SUBARU", "SUUBA": "SUBARU", "SUBR": "SUBARU", "SUZUK": "SUZUKI", "SUZUKI": "SUZUKI", "SUKI": "SUZUKI", "MITI": "MITSUBISHI", "MITS": "MITSUBISHI", "MITSUBISHI": "MITSUBISHI",

    # Korean
    "HYUN": "HYUNDAI", "HYUNDAI": "HYUNDAI", "HYUYN": "HYUNDAI", "HYUNI": "HYUNDAI", "HY": "HYUNDAI", "KIA": "KIA", "KIA.": "KIA", "OKIA": "KIA", "KI/AM": "KIA", "KI/CO": "KIA", "KIA M": "KIA",

    # American
    "FORD": "FORD", "FORDF": "FORD", "CHEV": "CHEVROLET", "CHEVR": "CHEVROLET", "CHEVROLET": "CHEVROLET", "GM": "GENERAL MOTORS", "GMC": "GMC", "GMC.": "GMC", "CADIL": "CADILLAC", "CADILAC": "CADILLAC", "CADILLAC": "CADILLAC", "CHRY": "CHRYSLER", "CHRYS": "CHRYSLER", "CHRYSLER": "CHRYSLER", "DODGE": "DODGE", "RAM": "RAM", "JEEP": "JEEP", "BUIC": "BUICK", "BUICK": "BUICK", "SATURN": "SATURN", "Satur": "SATURN", "PONT": "PONTIAC", "PONTIAC": "PONTIAC", "LINCO": "LINCOLN", "LINCOLN": "LINCOLN", "LICOL": "LINCOLN", "LICOLN": "LINCOLN", "HUMME": "HUMMER", "HUMMER": "HUMMER",

    # European
    "BMW": "BMW", "BMY": "BMW", "BMLH": "BMW", "AUDI": "AUDI", "VOLKS": "VOLKSWAGEN", "VOLKSWAGEN": "VOLKSWAGEN", "VW": "VOLKSWAGEN", "VOXWA": "VOLKSWAGEN", "VLKWG": "VOLKSWAGEN", "VLKW": "VOLKSWAGEN", "VOLVO": "VOLVO", "VOLV": "VOLVO", "MERC": "MERCEDES-BENZ", "MB": "MERCEDES-BENZ", "BENTZ": "MERCEDES-BENZ", "BENZ": "MERCEDES-BENZ", "MERCEDES": "MERCEDES-BENZ", "JAGUA": "JAGUAR", "JAGUAR": "JAGUAR", "LAND": "LAND ROVER", "LNDRV": "LAND ROVER", "LR": "LAND ROVER", "RANGE": "LAND ROVER", "ROVER": "LAND ROVER", "PEUGEOT": "PEUGEOT", "FIAT": "FIAT", "ALFA": "ALFA ROMEO", "AL FA": "ALFA ROMEO", "SKODA": "SKODA", "RENAU": "RENAULT", "RENAULT": "RENAULT", "MINI": "MINI", "MINI-": "MINI",

    # Luxury/Other
    "LEXUS": "LEXUS", "LEXS": "LEXUS", "LEX": "LEXUS", "INFI": "INFINITI", "INFTI": "INFINITI", "INFTY": "INFINITI", "INFIN": "INFINITI", "INFM": "INFINITI", "TESL": "TESLA", "TESLA": "TESLA", "TESM": "TESLA", "TESMR": "TESLA", "LUCID": "LUCID", "LUCI": "LUCID", "PORSC": "PORSCHE", "PORS": "PORSCHE", "POR": "PORSCHE", "ASTO": "ASTON MARTIN", "BENTLEY": "BENTLEY", "BENTL": "BENTLEY", "ROLLS": "ROLLS ROYCE", "RR": "ROLLS ROYCE", "R ROV": "ROLLS ROYCE", "MAYBA": "MAYBACH", "MASER": "MASERATI", "MASE": "MASERATI", "MAZER": "MASERATI", "FERRA": "FERRARI", "FERRARI": "FERRARI", "LAMB": "LAMBORGHINI", "LAMBO": "LAMBORGHINI", "LAMBA": "LAMBORGHINI",

    # Commercial/Trucks
    "FREIG": "FREIGHTLINER", "FREIT": "FREIGHTLINER", "FREIGHTLINER": "FREIGHTLINER", "KENWO": "KENWORTH", "KENWORTH": "KENWORTH", "PETER": "PETERBILT", "PETE": "PETERBILT", "PTRBL": "PETERBILT", "INTL": "INTERNATIONAL", "INT": "INTERNATIONAL", "IC": "INTERNATIONAL", "ISUZU": "ISUZU", "ISUZ": "ISUZU", "HINO": "HINO", "MACK": "MACK", "MACKD": "MACK", "VOLVO": "VOLVO", "STERL": "STERLING", "STERLING": "STERLING", "WESTE": "WESTERN STAR", "WEST": "WESTERN STAR", "WST": "WESTERN STAR", "BLUEBIRD": "BLUE BIRD", "BLUBR": "BLUE BIRD", "THOMA": "THOMAS", "THOMAS": "THOMAS", "GILLIG": "GILLIG", "GILL": "GILLIG", "ICBUS": "IC BUS", "IC BUS": "IC BUS",

    # Motorcycles
    "YAMAH": "YAMAHA", "YAMAHA": "YAMAHA", "YAMAS": "YAMAHA", "KAWAS": "KAWASAKI", "KAWASAKI": "KAWASAKI", "HARLE": "HARLEY-DAVIDSON", "HARLEY": "HARLEY-DAVIDSON", "HODAK": "HODAKA", "DUC": "DUCATI", "DUCATI": "DUCATI", "KYMCO": "KYMCO", "VESPA": "VESPA", "PIAGG": "PIAGGIO", "PIAGGIO": "PIAGGIO",

    # Chinese/Other
    "BYD": "BYD", "GEELY": "GEELY", "CHERY": "CHERY", "DFSK": "DFSK", "WULING": "WULING", "BAIC": "BAIC",
}


In [0]:
# Convert the dictionary to key-value pairs for create_map
mapping_expr = create_map([lit(x) for x in chain(*vehicle_make_map.items())])

silver_df = silver_df.withColumn(
    "Vehicle_Make",
    coalesce(mapping_expr.getItem(upper(trim("Vehicle_Make"))), upper(trim("Vehicle_Make")))
)

In [0]:
silver_df.display()

In [0]:
# Show entire result (Optional) - If you want to see entire data

silver_df.select("vehicle make").show(100, truncate=False)

#### Mapping Registration State Name

In [0]:
distinct_registration_state = [row[0] for row in silver_df.select("Registration_State").distinct().collect()]
print(distinct_registration_state)

In [0]:
state_mapping = {
    "AZ": "Arizona", "SC": "South Carolina", "NS": "Nova Scotia", "LA": "Louisiana",
    "MN": "Minnesota", "NJ": "New Jersey", "DC": "District of Columbia", "OR": "Oregon",
    "99": "Unknown", "VA": "Virginia", "RI": "Rhode Island", "KY": "Kentucky", "WY": "Wyoming",
    "BC": "British Columbia", "NH": "New Hampshire", "MI": "Michigan", "NV": "Nevada",
    "GV": "Government", "QB": "Quebec", "WI": "Wisconsin", "ID": "Idaho", "CA": "California",
    "CT": "Connecticut", "NE": "Nebraska", "MT": "Montana", "NC": "North Carolina",
    "VT": "Vermont", "MD": "Maryland", "DE": "Delaware", "MO": "Missouri", "IL": "Illinois",
    "ME": "Maine", "MB": "Manitoba", "WA": "Washington", "ND": "North Dakota", "MS": "Mississippi",
    "AL": "Alabama", "IN": "Indiana", "OH": "Ohio", "TN": "Tennessee", "IA": "Iowa",
    "NM": "New Mexico", "PA": "Pennsylvania", "SD": "South Dakota", "FO": "Foreign",
    "NY": "New York", "ON": "Ontario", "SK": "Saskatchewan", "AB": "Alberta", "TX": "Texas",
    "WV": "West Virginia", "GA": "Georgia", "MA": "Massachusetts", "KS": "Kansas", "FL": "Florida",
    "CO": "Colorado", "AK": "Alaska", "AR": "Arkansas", "NB": "New Brunswick", "OK": "Oklahoma",
    "PR": "Puerto Rico", "UT": "Utah", "HI": "Hawaii", "DP": "Diplomat", "PE": "Prince Edward Island",
    "MX": "Mexico", "NF": "Newfoundland", "YT": "Yukon", "NT": "Northwest Territories"
}


In [0]:
#Mapping to Full state names

# Create mapping expression
mapping_expr = create_map([lit(x) for x in chain(*state_mapping.items())])

# Apply mapping to a correctly spelled and trimmed column
silver_df = silver_df.withColumn(
    "Registration_State",
    mapping_expr.getItem(upper(trim(col("Registration_State"))))
)

In [0]:
silver_df.select("Registration_State").display()

#### Mapping Plate Type

In [0]:
distinct_Plate_Type = [row.Plate_Type for row in silver_df.select("Plate_Type").distinct().collect()]
print(distinct_Plate_Type)

In [0]:
plate_type_map = {
    'CCK': 'Check',
    'OMO': 'Omnibus',
    'LMB': 'Limousine',
    'SOS': 'Special Service',
    'SPC': 'Special Plate',
    'SUP': 'Super Plate',
    'NYA': 'NY Authority',
    'APP': 'Applicant',
    'RGL': 'Regular',
    'STA': 'State',
    'CHC': 'Charter',
    'BOT': 'Boat',
    'RGC': 'Regional',
    'TRC': 'Truck',
    'AMB': 'Ambulance',
    'COM': 'Commercial',
    'BOB': 'Bobtail',
    'HAM': 'Hamlet',
    'CMH': 'Commercial Heavy',
    'NYS': 'New York State',
    'MCD': 'Motorcycle',
    'IRP': 'IRP Plate',
    'ORG': 'Organizational',
    '999': 'Unknown',
    'WUG': 'Wagon Utility',
    'LTR': 'Letter',
    'ORC': 'Official Race Car',
    'ITP': 'Interstate Plate',
    'SRF': 'Surfboard',
    'LUA': 'Luxury Auto',
    'USS': 'Utility Service',
    'AGR': 'Agricultural',
    'OML': 'Omni Line',
    'MCL': 'Motorcycle Club',
    'PAS': 'Passenger',
    'ATV': 'All-Terrain Vehicle',
    'OMR': 'Omni Regional',
    'OMT': 'Omni Transport',
    'OMV': 'Omni Van',
    'SCL': 'School',
    'PHS': 'Public Service',
    'GSM': 'Government Service',
    'STG': 'Stage',
    'HIR': 'Hire',
    'PSD': 'Police Department',
    'CSP': 'Commercial Service Plate',
    'AYG': 'Army Guard',
    'HIS': 'Historic',
    'MED': 'Medical',
    'LMA': 'Limited Mobility',
    'ARG': 'Agriculture',
    'DLR': 'Dealer',
    'SRN': 'Senior',
    'VPL': 'Van Pool',
    'SEM': 'Semi-Trailer',
    'SPO': 'Sports',
    'CMB': 'Combined',
    'TOW': 'Tow Truck',
    'VAS': 'Van Service',
    'NLM': 'Non-Licensed Motor',
    'TRL': 'Trailer',
    'OMS': 'Omni Service',
    'TRA': 'Trade',
    'JSC': 'Justice Service',
    'LMC': 'Limousine Company',
    'HOU': 'Household',
    'HSM': 'High School',
    'MOT': 'Motor',
    'JWV': 'Jewish War Veteran',
    'CBS': 'Cab Service',
    'NYC': 'New York City',
    'CLG': 'College',
    'FAR': 'Farm',
    'USC': 'United States Citizen',
    'AGC': 'Agency',
    'SNO': 'Snowmobile',
    'ORM': 'Ornamental',
    'TMP': 'Temporary',
    'GAC': 'Government Auto',
    'THC': 'Theatrical',
    'OMF': 'Omni Freight',
    'JCA': 'Judicial Court',
    'PPH': 'Public Passenger',
    'LOC': 'Local',
    'CME': 'Commercial Motor',
    'ATD': 'Attendant',
    'JCL': 'Judicial Clerk',
    'HAC': 'Hazardous Cargo',
    'GSC': 'Government Service Commercial',
}


In [0]:
# Convert dict to sequence of key, value pairs for create_map
mapping_expr = create_map([lit(x) for x in chain(*plate_type_map.items())])

silver_df = silver_df.withColumn(
    "Plate_Type",
    coalesce(mapping_expr[col("Plate_Type")], col("Plate_Type"))
)

In [0]:
silver_df.display(10)

#### Mapping for Borough/ Area (Violation County)

In [0]:
distinct_Violation_County = [row['Violation_County'] for row in silver_df.select("Violation_County").distinct().collect()]
print(distinct_Violation_County)

In [0]:

violation_county_mapping = {
    "K": "Kings", "Kings": "Kings", "KINGS": "Kings", "BK": "Kings", "K   F": "Kings",
    "Q": "Queens", "QN": "Queens", "QNS": "Queens", "Qns": "Queens", "QUEEN": "Queens",
    "BX": "Bronx", "Bronx": "Bronx", "BRONX": "Bronx",
    "R": "Richmond", "Rich": "Richmond", "RICH": "Richmond", "ST": "Richmond",
    "NY": "Manhattan", "MN": "Manhattan",
    "N": "Unknown", "VINIS": "Unknown", "108": "Unknown", "NEWY": "New York County", None: "Unknown"
}

In [0]:
# Create mapping expression
mapping_expr = create_map([lit(x) for x in chain(*violation_county_mapping.items())])

# Apply transformation to overwrite the same column
silver_df= silver_df.withColumn(
    "Violation_County",
    coalesce(mapping_expr.getItem(upper(trim("Violation_County"))), lit("Unknown"))
)

In [0]:
silver_df.display()

#### Normalizing Column Values (Violation Time)

In [0]:
from pyspark.sql.functions import col, regexp_extract, lpad, concat, lit, upper

# Extract hour, minute, and AM/PM
silver_df = silver_df.withColumn(
    "violation_hour",
    regexp_extract(col("violation_time"), r"^(\d{1,2})\d{2}[APap]$", 1)
).withColumn(
    "violation_minute",
    regexp_extract(col("violation_time"), r"^\d{1,2}(\d{2})[APap]$", 1)
).withColumn(
    "violation_ampm",
    upper(regexp_extract(col("violation_time"), r"^.*([APap])$", 1))
)

In [0]:
silver_df.display(100)

In [0]:
# Remove leading zero from hour for display (optional)
silver_df = silver_df.withColumn(
    "violation_hour_display",
    when(col("violation_hour").startswith("0"), col("violation_hour").substr(2, 1))
    .otherwise(col("violation_hour"))
)

In [0]:
# Build the formatted time string, e.g., "10:17 AM"
silver_df = silver_df.withColumn(
    "Parking_Violation_Time",
    concat(
        col("violation_hour_display"), lit(":"), col("violation_minute"), lit(" "),
        when(col("violation_ampm") == "A", lit("AM")).when(col("violation_ampm") == "P", lit("PM"))
    )
)



In [0]:
# Show result
silver_df.select("violation_time", "Parking_Violation_Time").display()

In [0]:
silver_df = silver_df.drop(
    "violation_time",
    "Parking Violation Time",
    "violation_hour",
    "violation_minute",
    "violation_ampm",
    "violation_hour_display"
)

In [0]:
silver_df.display(50)

#### Normalizing Values (Days Parking in Effect)

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def days_in_effect_full(code):
    if code is None:
        return None
    # Remove spaces for variable-length codes
    code = code.replace(" ", "")
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    # Pad to 7 chars
    code = code.ljust(7)
    result = [day for flag, day in zip(code, days) if flag.upper() == "Y"]
    return ", ".join(result) if result else "No days"

days_in_effect_full_udf = udf(days_in_effect_full, StringType())

silver_df = silver_df.withColumn(
    "Days_Parking_in_Full_Effect",
    days_in_effect_full_udf("Days_Parking_In_Effect")
)

In [0]:
# Show result
silver_df.select("Days_Parking_in_Full_Effect", "Days_Parking_In_Effect").display()

In [0]:
silver_df.drop("Days_Parking_In_Effect")

In [0]:
silver_df.coalesce(1).write.mode("overwrite").option("header", True).csv("/dbfs/FileStore/final_output")


###### Set up the ADLS credentials

In [0]:
spark.conf.set("fs.azure.account.key.<adls-storage-account-name>.dfs.core.windows.net", "<adls-access-keys>")

In [0]:
output_path = "abfss://<adls-storage-container-name>@<adls-storage-account-name>.dfs.core.windows.net/silver"

In [0]:
silver_df.write.mode("overwrite").parquet(output_path)